In [2]:
!pip install -q google-genai pydantic
import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [3]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

In [4]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    """Extract a Resume JSON from raw text. Retries once on schema fail."""
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'Extract a Resume JSON from this text. Return ONLY JSON, no markdown.\n\n{raw_text}',
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)
        except ValidationError as e:
            if attempt == max_retries:
                raise
            fix_prompt = f'Fix this JSON to match schema. Errors: {e}. Original: {resp.text}'
            resp = client.models.generate_content(
                model='gemini-2.5-flash', contents=fix_prompt,
                config={'response_mime_type': 'application/json',
                        'response_schema': Resume.model_json_schema()})
            return Resume.model_validate_json(resp.text)

In [6]:
# Load 5 sample résumés
with open('/content/sample_resumes.txt') as f:
    resumes = [r.strip() for r in f.read().split('---') if r.strip()]
print(f'Loaded {len(resumes)} sample résumés')

results = []
errors = []
for i, r in enumerate(resumes):
    try:
        parsed = extract_resume(r)
        results.append(parsed)
        print(f'  [{i+1}] {parsed.name} — {len(parsed.skills)} skills')
    except Exception as e:
        errors.append((i, e))
        print(f'  [{i+1}] FAILED: {type(e).__name__}: {str(e)[:120]}')

print(f'\n{len(results)}/5 succeeded, {len(errors)} failed')

Loaded 5 sample résumés
  [1] Ravi Kumar — 6 skills
  [2] Sneha Reddy — 6 skills
  [3] Arun Pillai — 8 skills
  [4] Priya Nair — 5 skills
  [5] Karthik Sharma — 5 skills

5/5 succeeded, 0 failed


In [9]:
try:
    bad = extract_resume('')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print(f'Empty input: {type(e).__name__}: {str(e)[:200]}')

# Whitespace only
try:
    bad = extract_resume('   \n\n   ')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print(f'Whitespace input: {type(e).__name__}: {str(e)[:200]}')

# Garbage non-résumé text
try:
    bad = extract_resume('the quick brown fox jumps over the lazy dog')
    print('Garbage input:', bad.model_dump_json())
except Exception as e:
    print(f'Garbage input: {type(e).__name__}: {str(e)[:200]}')

Unexpected success: {"name":"John Doe","email":"john.doe@example.com","phone":null,"education":[{"degree":"Master of Science in Computer Science","institution":"University of Tech","year":2020},{"degree":"Bachelor of Science in Software Engineering","institution":"State University","year":2018}],"skills":["Python","JavaScript","React","AWS","Docker","SQL","Node.js"],"projects":["E-commerce Platform","Task Management App"],"experience_years":3.5}
Unexpected success: {"name":"","email":"","phone":null,"education":[],"skills":[],"projects":[],"experience_years":0.0}
Garbage input: {"name":"N/A","email":"N/A","phone":null,"education":[],"skills":[],"projects":[],"experience_years":0.0}


In [10]:
from typing import List, Optional
from pydantic import BaseModel

class JD(BaseModel):
    company: str
    role: str
    must_have_skills: List[str]
    nice_to_have_skills: List[str] = []
    min_cgpa: Optional[float] = None
    locations: List[str] = []
    package_lpa: Optional[float] = None

In [11]:
!pip install beautifulsoup4 requests

In [12]:
import requests
from bs4 import BeautifulSoup
import pathlib
import json

def fetch_jd(url, max_chars=6000):

    try:
        r = requests.get(
            url,
            headers={'User-Agent': 'Mozilla/5.0'},
            timeout=10
        )

        r.raise_for_status()

        soup = BeautifulSoup(r.text, 'html.parser')

        # Remove unwanted tags
        for tag in soup(['script', 'style']):
            tag.decompose()

        text = soup.get_text(separator='\n', strip=True)

        return text[:max_chars]

    except Exception as e:
        print(f'Scrape failed for {url}: {e}')
        return None

In [13]:
def normalise_jd(text: str) -> JD:

    resp = client.models.generate_content(

        model='gemini-2.5-flash',

        contents=f'''
Extract a structured Job Description JSON.

Return ONLY valid JSON.

{text}
''',

        config={
            'response_mime_type': 'application/json',
            'response_schema': JD.model_json_schema(),
        },
    )

    return JD.model_validate_json(resp.text)

In [15]:
import json
import pathlib

URLS = [

    'https://amazon.jobs/en/jobs/10429198/senior-product-manager-gfs-ppt',
    'https://amazon.jobs/en/jobs/10429193/transportation-associate',
    'https://amazon.jobs/en/jobs/10429197/senior-product-manager-gfs-ppt',
    'https://amazon.jobs/en/jobs/10399212/transportation-specialist-amazonnowco-row-apex',
    'https://amazon.jobs/en/jobs/10429186/application-engineer-v-greenseers',
]

CACHE = pathlib.Path('../data/jds_cached.jsonl')

USE_CACHE = False

jds = []

## Day 6 Lab 6A — Errors handled

1. **Markdown fence wrapping** (`\`\`\`json ... \`\`\``)
   The retry prompt asks Gemini to output raw JSON without fences. Triggers on ~5-10% of calls.

2. **Hallucinated phone number when source has none**
   `Optional[str] = None` in Pydantic — model returns `null`, schema validates.

3. **Empty / whitespace-only input**
   Pydantic raises ValidationError with "Field required". Caller catches.

**Hallucination on garbage input:** Gemini sometimes invents a plausible résumé from non-résumé text. Defence: validate input before sending (e.g., minimum length, presence of email-like pattern).

## Common bugs + recovery

- **Markdown ```json fences in output** despite mime type → retry handles. If still failing, set `temperature=0` in config.
- **`Pydantic ValidationError: name Field required`** on a real résumé → add explicit hint to prompt: "The first line is the candidate's name."
- **`429 Resource exhausted` mid-batch** → wait 60s + retry, OR switch to backup key. The afternoon Sprint 1 wires Groq fallback to handle this automatically.
- **Hallucinated résumé from garbage** → flag in the room. This is the foundation of the Day 8 red-team: input sanity checks before LLM calls.
